# Image Acquisition

Acquire a full-frame image, a cropped rectangle scan, and images from multiple detectors.


### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run scripts/run_servers.py
```


### Imports


In [ ]:
import os
import json
import time

import tango
import numpy as np
import matplotlib.pyplot as plt
from tiled.client import from_uri

import sidpy

%matplotlib ipympl


### Ping servers


In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'microscope']

for name in server_names:
    device_name = f"asyncroscopy/{name}/default"
    proxy = tango.DeviceProxy(device_name)
    proxy.ping()
    print(device_name, proxy.state())


### Connect to devices


In [ ]:
scan = tango.DeviceProxy("asyncroscopy/scan/default")
microscope = tango.DeviceProxy("asyncroscopy/microscope/default")
data = tango.DeviceProxy("asyncroscopy/data/default")

# Backward-compatible aliases used by the workflow cells below.
mic_proxy = microscope
microscope_proxy = microscope

for proxy in (scan, microscope, data):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("data      :", data.state())


### Start Tiled data server


In [ ]:
TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
save_path = "D:/microscopedata/tiled/ahoust17/2026_05_22_AtomFab/"

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = save_path

if str(data.tiled_server).lower() != "yes":
    print("Tiled server is not responding; starting it from the DATA device...")
    config = json.loads(data.start_tiled_server())
else:
    print("Tiled server is already running.")
    config = json.loads(data.get_config())

print(json.dumps(config, indent=2))

client = from_uri(config.get("uri", f"http://{TILED_HOST}:{TILED_PORT}"))
print("Tiled keys:", list(client))


### Configure scan


In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 1024
scan.scan_region = [0, 0, 1, 1]

print("dwell_time :", scan.dwell_time)
print("image size :", scan.imsize)
print("scan region:", list(scan.scan_region))


### Acquire a HAADF image


In [ ]:
key = microscope.get_scanned_image()
node = client[key]
image = np.asarray(node.read())
metadata = dict(node.metadata)

print("Tiled key  :", key)
print("Metadata   :", metadata)
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)


### Display the image


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap="gray", interpolation="none")
ax.set_title(f"HAADF - dwell {scan.dwell_time * 1e6:.1f} us")
ax.axis("off")
plt.tight_layout()


### Acquire multiple detectors


In [ ]:
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0, 1, 1]

keys = json.loads(microscope.get_images(["HAADF", "BF"]))
images = []

for key in keys:
    node = client[key]
    img = np.asarray(node.read())
    metadata = dict(node.metadata)
    detector_name = metadata.get("detector", key)
    images.append((detector_name, key, img, metadata))
    print(detector_name, key, img.shape, img.dtype)


In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 5))
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, key, img, img_metadata) in zip(axes, images):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(str(detector_name).upper())
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()


### Acquire a cropped rectangle scan


In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0.4, 1, 0.2]

key = microscope.get_scanned_image_advanced()
node = client[key]
cropped_image = np.asarray(node.read())
cropped_metadata = dict(node.metadata)

print("Tiled key  :", key)
print("Metadata   :", cropped_metadata)
print("Image shape:", cropped_image.shape)
print("Image dtype:", cropped_image.dtype)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cropped_image, cmap="gray")
ax.set_title(f"Cropped scan: {key}")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
